In [14]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from functools import reduce
from typing import List, Tuple

In [15]:
output_path = "f0_OSMOSYS_ECU_Output.csv"
df = pd.read_csv(output_path)
ndc = pd.read_csv("docs/NDCOutput.csv")
planmicc = pd.read_csv("docs/PLANMICCOutput.csv")


In [16]:
initial_year = 2010
final_year = 2035

In [17]:
# Filter only co2e emissions 
def get_emissions_by_model(dataFrame: pd.DataFrame, emissions:List[str]) -> pd.DataFrame:
    emission_filters = (dataFrame['Year'] <= final_year) & (~dataFrame['Emission'].isna()) & (~dataFrame['AnnualEmissions'].isna()) & (dataFrame['Emission'].isin(emissions))
    emissions_output = dataFrame.loc[emission_filters, ['Strategy', 'Emission','Year', 'AnnualEmissions']]
    missing_emissions =  set(emissions) - set(emissions_output['Emission'].unique())
    if len(missing_emissions) > 0:
        print("WARNING: The following emissions are missing:")
        print(missing_emissions)
    emissions_output['AnnualEmissions'] = emissions_output['AnnualEmissions'] * 1000
    ippu_emissions = emissions_output.pivot_table(index = ['Year'], columns='Strategy' ,values = 'AnnualEmissions', aggfunc='sum').reset_index()
    return ippu_emissions

In [18]:
def draw_model_emission_scatter(name:str, dataFrame:pd.DataFrame) -> List[go.Scatter]:
    data = []
    for scenario in dataFrame.columns[1:]:
        data.append(go.Scatter(name=f"{scenario}({name})", x=dataFrame['Year'], y=dataFrame[scenario]))
    
    return data

# Fig. Emisiones Totales
def draw_emissions_trayectory(emissions:List[Tuple[str,pd.DataFrame]]):
    fig_data=[]
    for model in emissions:
        fig_data += draw_model_emission_scatter(model[0], model[1])
    
    fig = go.Figure(data=fig_data)


    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"Trayectoria Emisiones sector Procesos Industriales",
        },
        title_x=0.5,
        yaxis_title = "[GgCO2e]"
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Emisiones:</span> <b>%{y:.3f}</b>
    """)

    fig.show()

In [19]:
ndc_emissions = get_emissions_by_model(ndc, ['CO2e_HFC', 'CO2e_CEM'])
planmicc_emissions= get_emissions_by_model(planmicc, ['CO2e_CEM','CO2e_RESTO'])
df_emissions = get_emissions_by_model(df,['CO2e_OTHER', 'CO2e_HFC', 'CO2e_CEM'])
draw_emissions_trayectory([
    ('Current', df_emissions), 
    ('PLANMICC', planmicc_emissions), 
    ('NDC', ndc_emissions)
])


In [20]:
# Scenario

# current
current_scenarios = list(df['Strategy'].unique())

# planmicc
planmicc_scenarios = list(planmicc['Strategy'].unique())

# ndc
ndc_scenarios = list(ndc['Strategy'].unique())

In [21]:
def get_grouped_techs_emissions_by_column(dataFrame: pd.DataFrame,column:str, strategy:str, emission_labels:List[str], groupings:dict):
  fuel_groups = {x: k for k, v in groupings.items() for x in v}
  fuel_ippu_filters =  (dataFrame['Year'] <= final_year) & (~dataFrame['Emission'].isna()) & (~dataFrame[column].isna()) & (dataFrame['Strategy'] == strategy)
  
  if len(emission_labels) != 0:
    fuel_ippu_filters = fuel_ippu_filters & (dataFrame['Emission'].isin(emission_labels))

  fuel_ippu_output = dataFrame.loc[fuel_ippu_filters, ['Emission','Year', column]]
  fuel_ippu_output['Emission'] = fuel_ippu_output['Emission'].map(fuel_groups) # replace tech codes for group codes, ignore if not mapped
  fuel_ippu = fuel_ippu_output.pivot_table(index = ['Year'], columns='Emission' ,values = column, aggfunc='sum').reset_index()
  return fuel_ippu


def draw_tech_ippu_area_by_column(dataFrame: pd.DataFrame, strategy:str, title:str, y_label:str):
    # Fig. Emissions per Technology vs Year
    fig = px.area(dataFrame, 
                x="Year", 
                y=dataFrame.columns[1:], 
                labels={
                    "variable":"",
                    "Year": "Año",
                    "value": y_label
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"{title} - {strategy}",
        },
        title_x=0.5,
    )

    fig.update_layout(legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.3,     
            xanchor="right",
            x=1
    ))

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Valor:</span> <b>%{y:.3f}</b>
    """)
    fig.show()

In [22]:
# Current Emissions Disaggregated Emissions
emissions_techs_groups = {
  "Industria cementera": ["CO2e_CEM"],
  "Uso de HFC": ["CO2e_HFC"],
  "Industria no cementera sin HFC": ["CO2e_OTHER"],
}
# current

for scenario in current_scenarios:
    ippu_emission = get_grouped_techs_emissions_by_column(df,'AnnualEmissions', scenario, ['CO2e_CEM', 'CO2e_HFC', 'CO2e_OTHER'], emissions_techs_groups)
    draw_tech_ippu_area_by_column( ippu_emission, scenario ,"Emisiones desagregadas de procesos industriales (Current)", "[Mt CO2eq]")

# # planmicc

# for scenario in planmicc_scenarios:
#     ippu_emission = get_grouped_techs_emissions_by_column(planmicc,'AnnualEmissions', scenario, ['CO2e_CEM', 'CO2e_HFC', 'CO2e_OTHER'], emissions_techs_groups)
#     draw_tech_waste_by_column( ippu_emission, scenario ,"Emisiones desagregadas de procesos industriales (PLANMICC)", "[Mt CO2eq]")

# # ndc

# for scenario in ndc_scenarios:
#     ippu_emission = get_grouped_techs_emissions_by_column(ndc,'AnnualEmissions', scenario, ['CO2e_CEM', 'CO2e_HFC', 'CO2e_OTHER'], emissions_techs_groups)
#     draw_tech_waste_by_column( ippu_emission, scenario ,"Emisiones desagregadas de procesos industriales (NDC)", "[Mt CO2eq]")

In [23]:
def get_grouped_techs_emissions_by_column(dataFrame: pd.DataFrame,column:str, strategy:str, emission_label:str, groupings:dict):
  fuel_groups = {x: k for k, v in groupings.items() for x in v}
  fuel_ippu_filters =  (dataFrame['Year'] <= final_year) & (~dataFrame['Technology'].isna()) & (~dataFrame[column].isna()) & (dataFrame['Strategy'] == strategy)
  
  if emission_label != '' and emission_label != None:
    fuel_ippu_filters = fuel_ippu_filters & (dataFrame['Emission'] == emission_label)

  fuel_ippu_output = dataFrame.loc[fuel_ippu_filters, ['Technology','Year', column]]
  fuel_ippu_output['Technology'] = fuel_ippu_output['Technology'].map(fuel_groups) # replace tech codes for group codes, ignore if not mapped
  fuel_ippu = fuel_ippu_output.pivot_table(index = ['Year'], columns='Technology' ,values = column, aggfunc='sum').reset_index()
  return fuel_ippu


# Fig. Emisiones Totales
def draw_techs_production_line_by_strategy(dataFrame: pd.DataFrame, strategy:str, title:str, y_label:str):
    # Fig. Emissions per Technology vs Year
    fig = px.line(dataFrame, 
                x="Year", 
                y=dataFrame.columns[1:], 
                labels={
                    "variable":"",
                    "Year": "Año",
                    "value": y_label
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"{title} - {strategy}",
        },
        title_x=0.5,
    )

    fig.update_layout(legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.3,     
            xanchor="right",
            x=1
    ))

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Valor:</span> <b>%{y:.3f}</b>
    """)
    fig.show()



In [24]:
# Current Emissions Disaggregated Emissions
emissions_techs_groups = {
  "Producción de cemento": ["PROD_CEM"],
  "Producción de CLINKER": ["PROD_CLK_TRAD"],
}

# current

for scenario in current_scenarios:
    ippu_prod = get_grouped_techs_emissions_by_column(df,'ProductionByTechnology', scenario, '', emissions_techs_groups)
    draw_techs_production_line_by_strategy( ippu_prod, scenario ,"Trayectoria de produccion de cemento y de clinker (Current)", "[Mt CO2eq]")

# # planmicc

# for scenario in planmicc_scenarios:
#     ippu_prod = get_grouped_techs_emissions_by_column(planmicc,'ProductionByTechnology', scenario, '', emissions_techs_groups)
#     draw_techs_production_line_by_strategy( ippu_prod, scenario ,"Trayectoria de produccion de cemento y de clinker (PLANMICC)", "[Mt CO2eq]")

# # ndc

# for scenario in ndc_scenarios:
#     ippu_prod = get_grouped_techs_emissions_by_column(ndc,'ProductionByTechnology', scenario, '', emissions_techs_groups)
#     draw_techs_production_line_by_strategy( ippu_prod, scenario ,"Trayectoria de produccion de cemento y de clinker (NDC)", "[Mt CO2eq]")


In [25]:
def calc_clink_prod_factor(group):
  num = group.loc[group['Technology'] == 'PROD_CLK_TRAD', ['ProductionByTechnology']].iloc[0]
  den = group.loc[group['Technology'] == 'PROD_CEM', ['ProductionByTechnology']].iloc[0]
  return (num/den)*100

def get_clink_prod_factor(dataFrame: pd.DataFrame, strategy:str):
  fuel_ippu_filters =  (dataFrame['Year'] <= final_year) & (~dataFrame['Technology'].isna()) & (dataFrame['Technology'].isin(['PROD_CLK_TRAD', 'PROD_CEM'])) & (dataFrame['Strategy'] == strategy) & (~dataFrame['ProductionByTechnology'].isna())
  fuel_ippu_output = dataFrame.loc[fuel_ippu_filters, ['Year', 'Technology', 'ProductionByTechnology']]
  fuel_ippu = fuel_ippu_output.groupby('Year')[['Year', 'Technology', 'ProductionByTechnology']].apply(calc_clink_prod_factor).reset_index()
  fuel_ippu['other'] = 100 - fuel_ippu['ProductionByTechnology']
  return fuel_ippu

def draw_tech_factor(dataFrame:pd.DataFrame, title:str):
    # Fig. Emissions per Technology vs Year
    fig = px.area(dataFrame, 
                x="Year", 
                y=dataFrame.columns[1:],
                labels={
                    "variable":"",
                    "Year": "Año",
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":title
        },
        title_x=0.5,
        yaxis_title = "Porcentaje[%]",
    )

    fig.update_layout(showlegend=False)

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Porcentaje:</span> <b>%{y:.3f} %</b>
    """)
    fig.show()


In [26]:

# current

for scenario in current_scenarios:
    ippu_prod = get_clink_prod_factor(df, scenario)
    draw_tech_factor( ippu_prod ,f"Factor de clinker(Current) - {scenario}")

# # planmicc

# for scenario in planmicc_scenarios:
#     ippu_prod = get_clink_prod_factor(planmicc, scenario)
#     draw_tech_factor( ippu_prod ,f" Factor de clinker(PLANMICC) - {scenario}")

# # ndc

# for scenario in ndc_scenarios:
#     ippu_prod = get_clink_prod_factor(ndc, scenario)
#     draw_tech_factor( ippu_prod ,f"Factor de clinker (NDC) - {scenario}")